# 거래 분류 에이전트 외부 API 호출 테스트

카카오 로컬 · 네이버 지역 · OpenAI · Jev 4개를 한 셀씩 호출한다.

- 맨 위 셀부터 실행. 상호를 바꾸려면 `NAME`만 고치고 아래 셀을 다시 실행
- 커밋 전에는 출력 지우기 (검색 결과·모델 답이 파일에 남는다)

In [ ]:
import json
import os
import urllib.parse
import urllib.request

import pandas as pd
from dotenv import find_dotenv, load_dotenv

from spending_classifier.common import CATEGORIES  # 노트북을 scripts/ 에서 열어야 import 된다

load_dotenv(find_dotenv(usecwd=True))  # scripts/ 에서 위로 올라가며 .env 를 찾는다
NAME = "NICE*쿠팡"


def call(url, headers=None, body=None):
    """(상태 코드, 응답 JSON 또는 에러 앞 300자). 본문이 있으면 POST, 없으면 GET"""
    data = json.dumps(body).encode() if body is not None else None
    req = urllib.request.Request(url, data=data, headers={"Content-Type": "application/json", **(headers or {})})
    try:
        with urllib.request.urlopen(req, timeout=20) as res:
            return res.status, json.loads(res.read())
    except urllib.error.HTTPError as e:
        return e.code, e.read().decode(errors="replace")[:300]


print("키 있음:", {k: bool(os.getenv(k)) for k in ("KAKAO_REST_API_KEY", "NAVER_CLIENT_ID", "OPENAI_API_KEY", "TYPESAFE_API_KEY")})

## 1. 카카오 로컬 (키워드로 장소 검색)
볼 것: `category_name`이 우리 카테고리와 얼마나 비슷한지

In [ ]:
status, r = call(f"https://dapi.kakao.com/v2/local/search/keyword.json?query={urllib.parse.quote(NAME)}&size=5",
                 {"Authorization": f"KakaoAK {os.getenv('KAKAO_REST_API_KEY', '')}"})
print(status)
pd.DataFrame(r["documents"], columns=["place_name", "category_name", "address_name"]) if status == 200 else r

## 2. 네이버 지역 검색 (네이버 클라우드 플랫폼 · NAVER API HUB)
볼 것: `category` 형식, 카카오와 결과가 같은지 (`title`의 `<b>` 태그는 검색어 강조). 한 번에 최대 5건

In [ ]:
status, r = call(f"https://naverapihub.apigw.ntruss.com/search/v1/local?query={urllib.parse.quote(NAME)}&display=5",
                 {"X-NCP-APIGW-API-KEY-ID": os.getenv("NAVER_CLIENT_ID", ""),
                  "X-NCP-APIGW-API-KEY": os.getenv("NAVER_CLIENT_SECRET", "")})
print(status)
pd.DataFrame(r["items"], columns=["title", "category", "address"]) if status == 200 else r

## 3. OpenAI (키와 `LLM_MODEL` 이름 확인)
404 · `model_not_found`면 `.env`의 `LLM_MODEL` 이름이 틀린 것

In [ ]:
status, r = call("https://api.openai.com/v1/responses",
                 {"Authorization": f"Bearer {os.getenv('OPENAI_API_KEY', '')}"},
                 {"model": os.getenv("LLM_MODEL", ""), "input": "한 단어로 답해: 안녕", "max_output_tokens": 50})
print(status, os.getenv("LLM_MODEL"))
{"model": r.get("model"), "output": r.get("output")} if status == 200 else r

## 4. Jev `Choice` (4단계 판단 후보)
볼 것: `choice`가 맞는지, `confidence`가 얼마나 나오는지. 지금은 상호 이름만 넣는다 → 다음에 검색 근거를 `state`에 같이 넣고 비교

In [ ]:
from typesafe_sdk import Choice, TypeSafeClient

with TypeSafeClient(model=os.getenv("JEV_MODEL")) as client:  # SDK 기본값은 TYPESAFE_DEFAULT_MODEL이라 직접 넘김
    res = client.system_one(
        state = f"가맹점 이름: {NAME})",
        questions={"category": Choice(instructions="이 가맹점의 소비 카테고리는?", criteria=CATEGORIES)},
    )
print(res.model)
res.answers["category"].model_dump()

## 5. 지저분한 이름 여러 개 한 번에 비교
4단계로 오는 건 SVM도 헷갈린 이름이다. 이름마다 SVM · 검색 끝 칸 · Jev(이름만 / 이름+카카오 끝 칸)를 한 줄로 본다.
볼 것: 검색 0건 비율, 검색을 넣으면 Jev 답이 맞게 바뀌는지, 틀린 답일 때 Jev 확률이 낮은지

In [ ]:
from pathlib import Path

import joblib
from typesafe_sdk import Choice, TypeSafeClient

from spending_classifier.common import normalize_name

ROOT = Path(find_dotenv(usecwd=True)).parent  # 저장소 루트 (.env 가 있는 곳)
SVM_PATH = ROOT / "local_workspace/models/spending_classifier/artifacts/content_policy_svm/spending_category_svm.joblib"  # 혜리 님 공유 파일, Git 제외
NAMES = ["(주)우아한형제들", "비바리퍼블리카", "NICE*쿠팡", "KCP*무신사", "스타벅스 강남역점",
         "GS25 역삼점", "새봄동물", "행복약국", "우리식당"]

svm = joblib.load(SVM_PATH)["estimator"]


def svm_top(name):
    p = svm.predict_proba([normalize_name(name)])[0]
    return f"{svm.classes_[p.argmax()]} {p.max():.2f}"


def kakao_leaf(name):
    s, r = call(f"https://dapi.kakao.com/v2/local/search/keyword.json?query={urllib.parse.quote(name)}&size=1",
                {"Authorization": f"KakaoAK {os.getenv('KAKAO_REST_API_KEY', '')}"})
    if s != 200:
        return None, f"에러 {s}"
    return (r["documents"][0]["category_name"].split(" > ")[-1], r["documents"][0]["place_name"]) if r["documents"] else (None, "0건")


def naver_leaf(name):
    s, r = call(f"https://naverapihub.apigw.ntruss.com/search/v1/local?query={urllib.parse.quote(name)}&display=1",
                {"X-NCP-APIGW-API-KEY-ID": os.getenv("NAVER_CLIENT_ID", ""), "X-NCP-APIGW-API-KEY": os.getenv("NAVER_CLIENT_SECRET", "")})
    if s != 200:
        return f"에러 {s}"
    return r["items"][0]["category"].split(">")[-1] if r["items"] else "0건"


def jev(client, state):
    a = client.system_one(state=state, questions={"c": Choice(instructions="이 가맹점의 소비 카테고리는?", criteria=CATEGORIES)}).answers["c"]
    return f"{a.choice} {a.confidence:.2f}"


rows = []
with TypeSafeClient(model=os.getenv("JEV_MODEL")) as client:
    for name in NAMES:
        leaf, found = kakao_leaf(name)
        rows.append({"이름": name, "SVM": svm_top(name), "카카오 1등": found, "카카오 끝 칸": leaf, "네이버 끝 칸": naver_leaf(name),
                     "Jev 이름만": jev(client, f"가맹점 이름: {name}"),
                     "Jev +업종": jev(client, f"가맹점 이름: {name} / 검색 업종: {leaf}") if leaf else "-"})
pd.DataFrame(rows)
